<a href="https://colab.research.google.com/github/MuhammadRabees/AI_ML-Internship-Advanced-Task-1/blob/main/Advanced_Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem Statement:** With the massive volume of news generated daily, manually sorting and categorizing articles is highly inefficient. There is a need for an automated, context-aware system capable of reading news headlines and instantly classifying them into relevant topics to streamline content delivery.


**Objective:** The primary goal is to fine-tune a pre-trained transformer model (BERT) to accurately classify news headlines into topic categories. Specifically, this project involves:

* Tokenizing and preprocessing the AG News Dataset.

* Fine-tuning the bert-base-uncased model for sequence classification.

* Evaluating the model's performance using Accuracy and F1-score metrics.

* Deploying the final model via a Gradio web interface for live, interactive text classification.




**Step 1: Install Dependencies**

In [1]:
!pip install transformers datasets evaluate scikit-learn gradio accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


**Step 2: Load Dataset**

In [2]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


**Step 3: Tokenize the Data**

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    # Padding and truncation ensure all sequences are the same length
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

**Step 4: Set Up the Model & Metrics**

In [4]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Map the numeric labels to actual category names for cleaner output later
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

# Taking a smaller subset of the data to speed up training
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

# Load evaluation metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Step 5: Train the Model**

In [5]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Evaluates at the end of each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

# Start training!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.405773,0.880000,0.880163
2,No log,0.370086,0.886000,0.886344
3,No log,0.371646,0.890000,0.890227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.37182718912760415, metrics={'train_runtime': 625.6389, 'train_samples_per_second': 9.59, 'train_steps_per_second': 0.599, 'total_flos': 1578694680576000.0, 'train_loss': 0.37182718912760415, 'epoch': 3.0})

**Step 6: Evaluating model using accuracy and F1 score**

In [6]:
results = trainer.evaluate()
print(f"Final Model Evaluation: {results}")

Final Model Evaluation: {'eval_loss': 0.37164589762687683, 'eval_accuracy': 0.89, 'eval_f1': 0.8902270662582443, 'eval_runtime': 17.1619, 'eval_samples_per_second': 29.134, 'eval_steps_per_second': 1.865, 'epoch': 3.0}


**Step 7: Deploy with Gradio**

In [ ]:
import gradio as gr
from transformers import pipeline

# Create a text-classification pipeline using your fine-tuned model
# device=0 ensures it uses the Colab GPU for faster inference
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

def predict_news(text):
    result = classifier(text)[0]
    return f"Predicted Category: {result['label']} (Confidence: {result['score']:.2f})"

# Build the Gradio interface
iface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=5, placeholder="Enter a news headline or snippet here..."),
    outputs="text",
    title="News Topic Classifier Using BERT",
    description="Classifies news into World, Sports, Business, or Sci/Tech."
)

# Launch it!
iface.launch()

**Final Summary / Insights**
                                                      
**Model Performance:** The bert-base-uncased model was successfully fine-tuned on a subset of the AG News dataset. After 3 epochs, the model achieved an impressive Evaluation Accuracy of 88.6% and an F1-Score of 88.6%.

**Key Insights:**

* **Efficiency of Transfer Learning**: Achieving nearly 89% accuracy on a drastically reduced dataset (2,000 training samples) highlights the strength of pre-trained transformer models. The model effectively leveraged its foundational language understanding to quickly adapt to this specific classification task.

* **Balanced Predictions:** The close alignment between the Accuracy and F1-score indicates that the model handles all four classes (World, Sports, Business, Sci/Tech) fairly equally, without severe bias toward a single majority class.

* **Real-Time Deployment:** Integrating Gradio directly into the workflow proved highly effective for rapid prototyping. It bridged the gap between raw backend metrics and an interactive, user-facing application